In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import time
from scipy.io import loadmat
from scipy import stats, signal
from scipy.sparse import csr_matrix, vstack, hstack, issparse
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')


# Set style for publication-quality plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")


In [3]:
from imports import *
from src.utils import poisson_glm_utils
from config import dir_config

from config.poisson_glm_config import StateBasedPoissonGLMConfig

In [4]:
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)
output_folder_name = 'equal_block_cross_validation_1coh_50choice'

session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, 'sessions_metadata.csv'))
session_metadata = session_metadata[~np.isin(session_metadata["session_id"], session_to_exclude)]

neuron_metadata = pd.read_csv(Path(processed_dir, 'neuron_metadata.csv'))
neuron_metadata = neuron_metadata[~np.isin(neuron_metadata["session_id"], session_to_exclude)].reset_index()

with open(Path(processed_dir, f'glm_hmm_models', f'glm_hmm_masked_final.pkl'), 'rb') as f:
    glm_hmm = pickle.load(f)

poisson_glm_config = StateBasedPoissonGLMConfig()

In [5]:
# Feature indices for easy access
feature_idx = {
    'target_start': 0,
    'target_end': poisson_glm_config.FEATURES_TARGET,
    'stim_start': poisson_glm_config.FEATURES_TARGET,
    'stim_end': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS,
    'saccade_start': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS,
    'saccade_end': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURES_SACCADE,
    'history_start': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURES_SACCADE,
    'history_end': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURES_SACCADE + poisson_glm_config.FEATURES_HISTORY,
    'intercept_idx': poisson_glm_config.get_total_features() - 1
}

print(f"Total features: {poisson_glm_config.get_total_features()}")

Total features: 85


In [6]:
assert poisson_glm_config.N_COHERENCE_LEVELS == 1, "Coherence levels are not set 1"

In [29]:
def extract_neuron_data(neuron_id, prior_cond, outcome_filter = "correct_only"):
    session_name = neuron_metadata.loc[neuron_metadata["neuron_id"] == neuron_id, "session_id"].values[0]
    data_path = Path(compiled_dir, session_name)

    # Load neural and behavioral data
    try:
        spike_times = np.load(data_path / "spike_times.npy")
        spike_clusters = np.load(data_path / "spike_clusters.npy")
    except:
        spike_times = loadmat(Path(compiled_dir, session_name, "spike_times.mat"))
        spike_times = spike_times["spike_times"][0]
        spike_clusters = loadmat(Path(compiled_dir, session_name, "spike_clusters.mat"))
        spike_clusters = spike_clusters["spike_clusters"][0]

    # Get neuron spike times
    cluster_id = neuron_metadata.cluster[neuron_metadata["neuron_id"] == neuron_id].values[0]
    neuron_spike_times = spike_times[spike_clusters == cluster_id]
    neuron_spike_times = (neuron_spike_times / 30).round().astype(int)  # Convert to ms

    # Get timestamps and trial data
    timestamps = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_timestamps.csv"), index_col=None)
    trial_info = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_trial.csv"), index_col=None)

    # Process trial data
    GP_trial_data = trial_info[trial_info.task_type == 1].reset_index(drop=True)
    # signed coherence
    GP_trial_data["signed_coherence"] = GP_trial_data["coherence"] * (2*GP_trial_data["target"]-1)

    GP_trial_data = GP_trial_data[GP_trial_data.reaction_time.notna()]
    # include equal block only
    if prior_cond == "equal_only":
        GP_trial_data = GP_trial_data[GP_trial_data.prob_toRF == 50]
        GP_trial_data["state"] = 0
    elif prior_cond == "unequal_only":
        GP_trial_data = GP_trial_data[GP_trial_data.prob_toRF != 50]
        GP_trial_data["state"] = 0
    else:
        raise NotImplementedError("Multiple blocks not implemented yet.")

    coh_levels = np.sort(GP_trial_data['signed_coherence'].unique()) /100  # Normalize coherence

    if outcome_filter == "correct_only":
        GP_trial_data = GP_trial_data[GP_trial_data.outcome == 1].reset_index()
    elif outcome_filter == "incorrect_only":
        GP_trial_data = GP_trial_data[GP_trial_data.outcome == 0].reset_index()

    return GP_trial_data, neuron_spike_times, timestamps, coh_levels


def create_neuroglm_trials(session_data, timestamps, neuron_spike_times, bin_size=1.0):
    """
    Create trial structure following neuroGLM format.
    """
    trials = []

    # Convert timestamps to ms
    timestamps_ms = (timestamps / 30).round()

    for idx, row in session_data.iterrows():
        trial_idx = row.trial_number - 1  # Convert to 0-based

        # Trial timing (relative to target onset - 50ms)
        target_onset = timestamps_ms.loc[trial_idx, "target_onset"]
        trial_start = target_onset - 50
        trial_end = timestamps_ms.loc[trial_idx, "response_onset"]
        duration = trial_end - trial_start # -50ms of target onset to response onset

        if pd.isna(duration) or duration <= 0:
            continue

        # Get trial spike times (relative to trial start)
        trial_spikes = neuron_spike_times[
            (neuron_spike_times >= trial_start) &
            (neuron_spike_times <= trial_end)
        ] - trial_start

        # Create binned spike train
        n_bins = int(np.ceil(duration / bin_size))
        spike_train = np.zeros(n_bins)

        for spike_time in trial_spikes:
            bin_idx = int(np.floor(spike_time / bin_size))
            if 0 <= bin_idx < n_bins:
                spike_train[bin_idx] += 1

        # Event timings (relative to trial start)
        events = {
            'target_onset': 50,  # Always 50ms into trial
            'stimulus_onset': timestamps_ms.loc[trial_idx, "stimulus_onset"] - trial_start,
            'stimulus_offset': timestamps_ms.loc[trial_idx, "response_onset"] - trial_start,
            'response_onset': timestamps_ms.loc[trial_idx, "response_onset"] - trial_start,
        }

        # Trial structure
        trial = {
            'duration': duration,
            'spike_train': spike_train,
            'n_bins': n_bins,

            # Event timings
            'target_onset': events['target_onset'],
            'stimulus_onset': events['stimulus_onset'],
            'stimulus_offset': events['stimulus_offset'],
            'response_onset': events['response_onset'],

            # Experimental variables
            'coherence': row.signed_coherence / 100,  # Normalize coherence
            'choice': row.choice,
            'state': row.state,
            'reaction_time': row.reaction_time,

            # Trial metadata
            'trial_idx': int(trial_idx),
        }

        trials.append(trial)

    return pd.DataFrame(trials)


def build_design_matrix(trials, coh_levels):
    # Initialize containers
    trial_matrices = []
    trial_spike_trains = []

    # Process each trial
    for trial in trials.itertuples():
        trial_duration = int(trial.duration)
        trial_design = np.zeros((trial_duration, poisson_glm_config.get_total_features()))

        # 1. TARGET ONSET COMPONENT
        target_bin = int(trial.target_onset)
        if 0 < target_bin <= trial_duration:
            target_matrix = np.zeros((trial_duration, 1))
            target_matrix[target_bin-1] = 1.0
            target_conv, _ = poisson_glm_utils.convolve_with_basis(
                target_matrix,
                poisson_glm_config.TARGET_BASIS,
                poisson_glm_config.TARGET_DURATION_MS,
                poisson_glm_config.TARGET_SPACING_MS,
                effect= poisson_glm_config.TARGET_EFFECT
            )

            target_start = feature_idx['target_start'] + int(trial.state) * poisson_glm_config.TARGET_N_BASES
            target_end = target_start + poisson_glm_config.TARGET_N_BASES
            trial_design[:, target_start:target_end] = target_conv


        # 2. STIMULUS COHERENCE COMPONENT
        stim_bin = int(trial.stimulus_onset)
        resp_bin = int(trial.response_onset)
        if 0 < stim_bin < resp_bin <= trial_duration:
            stim_matrix = np.zeros((trial_duration, 1))
            stim_matrix[stim_bin-1:resp_bin] = 1.0
            stim_conv, _ = poisson_glm_utils.convolve_with_basis(
                stim_matrix,
                poisson_glm_config.STIMULUS_BASIS,
                poisson_glm_config.STIMULUS_DURATION_MS,
                poisson_glm_config.STIMULUS_SPACING_MS,
                effect= poisson_glm_config.STIMULUS_EFFECT
            )

            # Assign to coherence-specific and state-specific features
            if poisson_glm_config.N_COHERENCE_LEVELS == 1:
                coh_idx = 0 # same stimulus kernel for all coherence levels (1stimulus)
            else:
                coh_idx = np.where(coh_levels == trial.coherence)[0][0]
            state_idx = int(trial.state)
            coh_start = feature_idx['stim_start'] + coh_idx * poisson_glm_config.STIMULUS_N_BASES + state_idx * poisson_glm_config.STIMULUS_N_BASES * len(coh_levels)
            coh_end = coh_start + poisson_glm_config.STIMULUS_N_BASES
            trial_design[:, coh_start:coh_end] = stim_conv

        # 3. SACCADE/CHOICE COMPONENT
        if 0 < resp_bin <= trial_duration:
            saccade_matrix = np.zeros((trial_duration, 1))
            saccade_matrix[resp_bin-1] = 1.0
            saccade_conv, _ = poisson_glm_utils.convolve_with_basis(
                saccade_matrix,
                poisson_glm_config.SACCADE_BASIS,
                poisson_glm_config.SACCADE_DURATION_MS,
                poisson_glm_config.SACCADE_SPACING_MS,
                effect= poisson_glm_config.SACCADE_EFFECT
            )

            # Assign to choice-specific features
            choice_idx = int(trial.choice)
            state_idx = int(trial.state)
            choice_start = feature_idx['saccade_start'] + choice_idx * poisson_glm_config.SACCADE_N_BASES + state_idx * poisson_glm_config.SACCADE_N_BASES * poisson_glm_config.N_CHOICE_OPTIONS
            choice_end = choice_start + poisson_glm_config.SACCADE_N_BASES
            trial_design[:, choice_start:choice_end] = saccade_conv


        # 4. POST-SPIKE HISTORY COMPONENT
        history_matrix = poisson_glm_utils.create_post_spike_history_matrix(trial.spike_train)
        trial_design[:, feature_idx['history_start']:feature_idx['history_end']] = history_matrix

        # 5. INTERCEPT TERM
        trial_design[:, feature_idx['intercept_idx']] = 1.0

        # Store processed trial
        trial_matrices.append(csr_matrix(trial_design))
        trial_spike_trains.append(trial.spike_train)

    X = vstack(trial_matrices, format='csr')
    y = np.concatenate(trial_spike_trains)

    return X, y



def fit_poisson_glm(X, y):
    if issparse(X):
        X = X.toarray() # Convert to dense for faster computation in this case
    # Feature standardization for better conditioning
    X_means = X.mean(axis=0)
    X_stds = X.std(axis=0)

    # Avoid division by zero
    X_stds[X_stds < 1e-8] = 1.0

    # Standardize all features except intercept
    X_scaled = X.copy()
    X_scaled[:, :-1] = (X[:, :-1] - X_means[:-1]) / X_stds[:-1]

    def loss_fun(w):
        eta = X_scaled @ w
        if np.any(y[eta < -15]>0):
            return 1e20  # Penalty if rate is 0 but spikes are present
        else:
            eta = np.clip(eta, -15, 15)  # Prevent overflow
            mu = np.exp(eta)
            return np.sum(mu) - np.dot(y, eta) + 0.1 * np.dot(w, w)

    def grad_fun(w):
        eta = X_scaled @ w
        eta = np.clip(eta, -15, 15)
        mu = np.exp(eta)
        return X_scaled.T @ (mu - y) + 0.02 * w

    n_features = X_scaled.shape[1]
    w_init = np.zeros(n_features)
    w_init[-1] = np.log(max(y.mean(), 1e-8))  # Smart intercept

    start_time = time.time()

    result = minimize(
        fun=loss_fun,
        x0=w_init,
        method='L-BFGS-B',
        jac=grad_fun,
        options={
            'maxiter': 1000,    # Very limited iterations
            'gtol': 1e-3,      # Relaxed tolerance
            'ftol': 1e-5,      # Relaxed tolerance
            'maxfun': 200      # Limit function calls
        }
    )

    fit_time = time.time() - start_time

    if result.success or result.fun < 1e6:
        weights_scaled = result.x

        # predicted y
        eta = X_scaled @ weights_scaled
        predicted = np.exp(np.clip(eta, -15, 15))
        result['predicted_y'] = predicted
    else:
        print(f"Optimization failed: {result.message}")
        return None

    return result


def predict_poisson_glm(X, model):
    if issparse(X):
        X = X.toarray()
    X_means = X.mean(axis=0)
    X_stds = X.std(axis=0)

    # Avoid division by zero
    X_stds[X_stds < 1e-8] = 1.0

    X_scaled = X.copy()
    X_scaled[:, :-1] = (X[:, :-1] - X_means[:-1]) / X_stds[:-1]

    eta = X_scaled @ model.x
    return np.exp(np.clip(eta, -15, 15))


## Load and prepare data

In [8]:
config_path = Path(processed_dir, 'poisson_glm', output_folder_name, 'config.json')
import os

config_path.parent.mkdir(parents=True, exist_ok=True)
poisson_glm_config.save(config_path)

In [ ]:
poissonglm_data_eq_corr = {}

for prior_cond in ["equal_only", "unequal_only"]:
    for outcome_filter in ["correct_only", "all"]:

        # create folder for storing results if it doesn't exist
        output_dir = Path(processed_dir, 'poisson_glm', "data", f"prior_cond_{prior_cond}_outcome_{outcome_filter}")
        output_dir.mkdir(parents=True, exist_ok=True)

        for neuron_id in neuron_metadata.neuron_id.unique():
            session_data, neuron_spike_times, timestamps, coh_levels = extract_neuron_data(neuron_id, prior_cond=prior_cond, outcome_filter=outcome_filter)
            trials_df = create_neuroglm_trials(session_data, timestamps, neuron_spike_times)
            # save trials_df for later use
            trials_df.to_parquet(output_dir / f"{neuron_id}.parquet", index=False)


### Cross-validation

In [15]:
    # loaded_trials_df = pd.read_parquet(output_dir / f"{neuron_id}.parquet")
    # build_design_matrix(loaded_trials_df, coh_levels)

1694

In [47]:
kf = KFold(n_splits=5, shuffle=True, random_state=216)

prior_cond = "equal_only"
outcome_filter = "correct_only"
poisson_glm_data_dir = Path(processed_dir, 'poisson_glm', "data", f"prior_cond_{prior_cond}_outcome_{outcome_filter}")

model_result_dir = Path(processed_dir, 'poisson_glm', "models", f"prior_cond_{prior_cond}_outcome_{outcome_filter}")
model_result_dir.mkdir(parents=True, exist_ok=True)

parquet_files_with_numbers = sorted(
    [(int(f.stem), f) for f in poisson_glm_data_dir.glob("*.parquet") if f.stem.isdigit()],
    key=lambda x: x[0]  # sort by the numeric value
)

for neuron_id, fpath in parquet_files_with_numbers:
    df = pd.read_parquet(fpath)
    coh_levels = np.sort(df['coherence'].unique()/100)

    fitting_result = {
            "coh_levels": coh_levels,
            "folds": []
        }

    for fold, (train_idx, test_idx) in enumerate(kf.split(df)):
        # print(f"Processing Neuron {neuron_id}, Fold {fold+1}/5")
        train_trials = df.iloc[train_idx]
        test_trials = df.iloc[test_idx]

        X_train, y_train = build_design_matrix(train_trials, coh_levels)
        X_test, y_test = build_design_matrix(test_trials, coh_levels)

        # fit model
        model_result = fit_poisson_glm(X_train, y_train)
        if model_result is None:
            print(f"Failed to fit model for Neuron {neuron_id}, Fold {fold+1}/5")
            break

        train_preds, onset = [], 0
        for t in train_trials.itertuples():
            n = t.n_bins
            train_preds.append(model_result["predicted_y"][onset:onset+n])
            onset += n

        test_flat_pred = predict_poisson_glm(X_test, model_result)
        test_preds, onset = [], 0
        for t in test_trials.itertuples():
            n = t.n_bins
            test_preds.append(test_flat_pred[onset:onset+n])
            onset += n

        # -----------------------
        # STORE
        # -----------------------
        fitting_result["folds"].append({
            "fold": fold,
            "train_data": pd.DataFrame(train_trials),
            "test_data": pd.DataFrame(test_trials),
            "model": model_result,
            "train_predictions": train_preds,
            "test_predictions": test_preds
        })

    break
    with open(Path(model_result_dir, f"neuron_{neuron_id}"), 'wb') as f:
        pickle.dump(fitting_result, f)


In [46]:
train_trials

,duration,spike_train,n_bins,target_onset,stimulus_onset,stimulus_offset,response_onset,coherence,choice,state,reaction_time,trial_idx
0,1694.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1694,50,743.0,1694.0,1694.0,-0.20,0.0,0.0,926.0,33
1,1369.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1369,50,903.0,1369.0,1369.0,0.50,1.0,0.0,446.0,34
2,1852.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1852,50,1330.0,1852.0,1852.0,0.20,1.0,0.0,498.0,36
4,1234.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1234,50,810.0,1234.0,1234.0,0.20,1.0,0.0,399.0,38
5,1499.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1499,50,1063.0,1499.0,1499.0,0.00,1.0,0.0,415.0,40
6,1596.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1596,50,1290.0,1596.0,1596.0,0.50,1.0,0.0,278.0,41
7,1262.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1262,50,956.0,1262.0,1262.0,-0.20,0.0,0.0,275.0,42
8,1571.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1571,50,1155.0,1571.0,1571.0,0.00,1.0,0.0,383.0,44
9,1124.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1124,50,757.0,1124.0,1124.0,-0.50,0.0,0.0,336.0,46
10,1208.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1208,50,836.0,1208.0,1208.0,0.06,1.0,0.0,352.0,47
